In [1]:
import pandas as pd
import numpy as np
calendar = pd.read_csv("calendar_processed.csv",parse_dates=['date'])
sales_train = pd.read_csv('sales_train.csv',parse_dates=['date'])
sales_test = pd.read_csv('sales_test.csv',parse_dates=['date'])
inventory = pd.read_csv('inventory.csv')

## 合并数据

In [2]:
#合并 sales train, sales test与inventory
inventory_subset = inventory[['unique_id', 'product_unique_id', 'warehouse']]

# Merge with sales_train and sales_test
sales_train = pd.merge(sales_train, inventory_subset, how='left', on=['unique_id', 'warehouse'])
sales_test = pd.merge(sales_test, inventory_subset, how='left', on=['unique_id', 'warehouse'])


In [3]:
#处理sales na step1:什么都有只没有sales
sales_train.loc[sales_train['total_orders'].notna() & sales_train['sell_price_main'].notna(), 'sales'] = sales_train['sales'].fillna(0)

## 小智helpme

In [6]:
#sales_test和sales_trains处理,drop availability,合并discounts,和calendar merge
def process_sales_data(sales_data, calendar):
    discount_cols = [f"type_{i}_discount" for i in range(7)]
    sales_data = pd.merge(sales_data, calendar, how='left', on=['date', 'warehouse'])
    sales_data["min_discount"] = sales_data[discount_cols].min(axis=1).fillna(1)
    
    # Drop discount columns
    drop_cols = discount_cols + ["availability"] if "availability" in sales_data.columns else discount_cols
    return sales_data.drop(columns=drop_cols, errors='ignore')

sales_train = process_sales_data(sales_train, calendar)
sales_test = process_sales_data(sales_test, calendar)

sales_test.to_csv('processed_sales_test.csv', index=False)

In [7]:
# One-hot encode the 'warehouse' column
encoded_warehouses = pd.get_dummies(sales_train['warehouse'], prefix='warehouse')
sales_train_encoded = pd.concat([sales_train.drop('warehouse', axis=1), encoded_warehouses], axis=1)

In [8]:
sales_train.columns

Index(['unique_id', 'date', 'warehouse', 'total_orders', 'sales',
       'sell_price_main', 'product_unique_id', 'holiday_name', 'shops_closed',
       'winter_school_holidays', 'school_holidays', 'year', 'month', 'day',
       'day_of_week', 'new_years_day', 'international_womens_day',
       'good_friday', 'holy_saturday', 'easter_day', 'easter_monday',
       'labour_day', 'mother's_day', 'cyrila_a_metodej', 'jan_hus',
       'den_ceske_statnosti', 'den_vzniku_samostatneho_ceskoslovenskeho_statu',
       'den_boje_za_svobodu_a_demokracii', 'christmas_eve',
       '1st_christmas_day', '2nd_christmas_day', 'den_osvobozeni',
       'memorial_day_of_the_republic',
       'memorial_day_for_the_victims_of_the_communist_dictatorships',
       'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday',
       'whit_monday', 'national_defense_day', 'day_of_national_unity',
       'independent_hungary_day', 'state_foundation_day',
       'memorial_day_for_the_martyrs_of_arad',
       'mem

In [9]:
sales_train

,unique_id,date,warehouse,total_orders,sales,sell_price_main,product_unique_id,holiday_name,shops_closed,winter_school_holidays,...,ascension_day,corpus_christi,german_unity_day,reformation_day,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday,min_discount
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,2375,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,2375,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,2375,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,2375,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,2375,good_friday,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,2023-06-21,Prague_1,9988.0,26.56,34.06,2422,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
4007415,4941,2023-06-24,Prague_1,8518.0,27.42,34.06,2422,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
4007416,4941,2023-06-23,Prague_1,10424.0,33.39,34.06,2422,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0
4007417,4941,2023-06-22,Prague_1,10342.0,22.88,34.06,2422,Not,0,0,...,0,0,0,0,0,0,0,1.0,1.0,0.0


## 继续处理sales na

In [10]:
#处理sales na step2:shop_closed == 1
# Fill sales with 0 if shops_closed == 1 and sales is NaN
sales_train.loc[(sales_train['shops_closed'] == 1) & (sales_train['sales'].isna()), 'sales'] = 0

# group sales_train by date, Fill missing price using the previous day's price within the same warehouse and unique_id
sales_train['sell_price_main'] = sales_train.groupby(['warehouse', 'unique_id'])['sell_price_main'].ffill()

In [11]:
missing_dates = sales_train[sales_train['sales'].isna()]['date'].unique()
missing_dates

array(['2021-05-21T00:00:00.000000000', '2021-05-22T00:00:00.000000000',
       '2021-12-10T00:00:00.000000000', '2021-12-09T00:00:00.000000000',
       '2021-06-27T00:00:00.000000000', '2021-06-26T00:00:00.000000000',
       '2021-05-29T00:00:00.000000000', '2021-05-31T00:00:00.000000000',
       '2021-05-30T00:00:00.000000000', '2021-06-20T00:00:00.000000000',
       '2021-06-12T00:00:00.000000000', '2021-07-04T00:00:00.000000000',
       '2021-06-19T00:00:00.000000000', '2021-06-13T00:00:00.000000000',
       '2021-07-11T00:00:00.000000000'], dtype='datetime64[ns]')

In [12]:
#处理sales na step3.5:shop_closed == 0
# Fill sales with 0 if shops_closed == 1 and sales is NaN

sales_train.loc[(sales_train['date'].isin(missing_dates)) & (sales_train['sales'].isna()), 'sales'] = 0

In [13]:
#处理sales na step4:疫情
# Get all relevant dates from missing_combinations
#missing_dates = # missing dates in sales_train

# Define the conditions for sales_train where sales should be filled
#condition_munich = (sales_train['warehouse'] == 'Munich_1') & (sales_train['date'].isin(missing_dates))
#condition_frankfurt = (sales_train['warehouse'] == 'Frankfurt_1') & (sales_train['date'].isin(missing_dates))

# Combine conditions
#condition = condition_munich | condition_frankfurt

# Compute the average sales for the same product_unique_id and date across other warehouses
#avg_sales = sales_train.groupby(['product_unique_id', 'date'])['sales'].transform(lambda x: x.mean())

# Fill sales where the condition is met and sales is NaN
#sales_train.loc[condition & sales_train['sales'].isna(), 'sales'] = avg_sales

In [14]:
#处理sales na step4:其他
# Step 2: For the rest of missing_dates in missing_combinations:
# - Set sales to 0
# - Fill total_orders using same day's value within the same warehouse, then use previous day's value if still missing
# - Fill price using previous day's price within the same warehouse and unique_id

# Identify remaining missing sales
remaining_condition = sales_train['date'].isin(missing_dates) & sales_train['sales'].isna()

# Fill sales with 0
sales_train.loc[remaining_condition, 'sales'] = 0

# Fill total_orders using same day's value within the same warehouse
sales_train['total_orders'] = sales_train.groupby(['date', 'warehouse'])['total_orders'].transform(lambda x: x.fillna(method='ffill'))

# Fill remaining missing total_orders using previous day's value within the same warehouse
sales_train['total_orders'] = sales_train.groupby('warehouse')['total_orders'].ffill()

# Fill price using previous day's price within the same warehouse and unique_id
sales_train['sell_price_main'] = sales_train.groupby(['warehouse', 'unique_id'])['sell_price_main'].ffill()

In [15]:
print(sales_train[sales_train['sales'].isna()])

Empty DataFrame
Columns: [unique_id, date, warehouse, total_orders, sales, sell_price_main, product_unique_id, holiday_name, shops_closed, winter_school_holidays, school_holidays, year, month, day, day_of_week, new_years_day, international_womens_day, good_friday, holy_saturday, easter_day, easter_monday, labour_day, mother's_day, cyrila_a_metodej, jan_hus, den_ceske_statnosti, den_vzniku_samostatneho_ceskoslovenskeho_statu, den_boje_za_svobodu_a_demokracii, christmas_eve, 1st_christmas_day, 2nd_christmas_day, den_osvobozeni, memorial_day_of_the_republic, memorial_day_for_the_victims_of_the_communist_dictatorships, memorial_day_for_the_victims_of_the_holocaust, whit_sunday, whit_monday, national_defense_day, day_of_national_unity, independent_hungary_day, state_foundation_day, memorial_day_for_the_martyrs_of_arad, memorial_day_of_the_1956_revolution, all_saints_day, hungary_national_day_holiday, christmas_holiday, 1848_revolution_memorial_day_(extra_holiday), all_saints'_day_holiday, a

In [16]:
sales_train.to_csv('processed_sales_train.csv', index=False)

In [24]:
# Step 1: Get all unique combinations of warehouse and date from the calendar
unique_combinations_calendar = calendar[['warehouse', 'date']].drop_duplicates()

# Step 2: Process each warehouse separately
result_frames = []

for warehouse in unique_combinations_calendar['warehouse'].unique():
    # Filter data for the current warehouse in calendar and sales_train
    calendar_data = unique_combinations_calendar[unique_combinations_calendar['warehouse'] == warehouse]
    warehouse_sales_data = sales_train[sales_train['warehouse'] == warehouse]

    # Determine the date range for this warehouse from sales_train
    start_date = warehouse_sales_data['date'].min()
    end_date = warehouse_sales_data['date'].max()

    # Filter calendar data to match the date range of sales_train
    warehouse_dates = calendar_data[(calendar_data['date'] >= start_date) & (calendar_data['date'] <= end_date)]['date'].unique()

    # Generate unique combinations of unique_id and date for this warehouse
    full_combinations = pd.MultiIndex.from_product(
        [warehouse_sales_data['unique_id'].unique(), warehouse_dates],
        names=['unique_id', 'date']
    ).to_frame(index=False)

    # Add the warehouse column to the full combinations
    full_combinations['warehouse'] = warehouse

    # Merge with the warehouse sales data
    merged_data = full_combinations.merge(
        warehouse_sales_data,
        on=['unique_id', 'warehouse', 'date'],
        how='left'
    )

    # Fill missing sales with 0
    merged_data['sales'].fillna(0, inplace=True)

    # Collect the result
    result_frames.append(merged_data)

    # Trigger garbage collection to free up memory
    #del full_combinations, warehouse_sales_data, merged_data
    #gc.collect()

# Step 3: Concatenate all results
sales_complete = pd.concat(result_frames)
sales_complete = sales_complete.reset_index(drop=True)

# Display the sorted result
print(sales_complete)


         unique_id       date warehouse  total_orders  sales  sell_price_main  \
0             1755 2020-08-01    Brno_1           NaN    0.0              NaN   
1             1755 2020-08-02    Brno_1           NaN    0.0              NaN   
2             1755 2020-08-03    Brno_1           NaN    0.0              NaN   
3             1755 2020-08-04    Brno_1           NaN    0.0              NaN   
4             1755 2020-08-05    Brno_1           NaN    0.0              NaN   
...            ...        ...       ...           ...    ...              ...   
7131293       1198 2024-05-29  Prague_3           NaN    0.0              NaN   
7131294       1198 2024-05-30  Prague_3           NaN    0.0              NaN   
7131295       1198 2024-05-31  Prague_3           NaN    0.0              NaN   
7131296       1198 2024-06-01  Prague_3           NaN    0.0              NaN   
7131297       1198 2024-06-02  Prague_3           NaN    0.0              NaN   

         product_unique_id 

In [25]:
sales_complete.to_csv('processed_sales_train_complete.csv', index=False)